# Random Forest Regressor — Ensemble Learning

## 1. Problem Setup

Assume we have a dataset

$$
\{(x_i, y_i)\}_{i=1}^{N}
$$

where

$$
x_i \in \mathbb{R}^D
$$

is the feature vector and

$$
y_i \in \mathbb{R}
$$

is a continuous target value.

The goal is to learn a function that predicts real-valued outputs.

---

## 2. Motivation

A single decision tree regressor has:

- **Low bias**
- **High variance**

Random Forest reduces variance by averaging multiple trees.

---

## 3. Bootstrap Sampling (Bagging)

For each tree $b = 1, \dots, B$:

Sample dataset with replacement:

$$
D^{(b)} \sim \text{Bootstrap}(D)
$$

Each tree is trained on a different dataset.

---

## 4. Feature Randomness

At each split, a random subset of features is selected:

$$
m = \lfloor \text{feature\_fraction} \cdot D \rfloor
$$

This reduces correlation between trees.

---

## 5. Tree Training

Each tree is trained independently:

$$
T_b = \text{DecisionTreeRegressor}(D^{(b)})
$$

Each tree uses:

- Random samples  
- Random feature subsets  
- A loss function (MSE, MAE or Huber)

---

## 6. Ensemble Model

The Random Forest consists of $B$ trees:

$$
\{T_1, T_2, \dots, T_B\}
$$

---

## 7. Prediction Rule (Averaging)

For a test sample $x$:

$$
\hat{y} = \frac{1}{B} \sum_{b=1}^{B} T_b(x)
$$

This reduces variance by averaging predictions.

---

## 8. Out-of-Bag (OOB) Samples

For each sample $i$, define:

$$
\mathcal{B}_i = \{b : x_i \notin D^{(b)}\}
$$

These are trees that did not see $x_i$.

---

## 9. OOB Prediction

For sample $i$:

$$
\hat{y}_i^{OOB} = \frac{1}{|\mathcal{B}_i|} \sum_{b \in \mathcal{B}_i} T_b(x_i)
$$

---

## 10. OOB Error (MSE)

The OOB estimate is:

$$
\text{MSE}_{OOB} =
\frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i^{OOB})^2
$$

---

## 11. Variance Reduction of Random Forest 

### I. Setup

Each tree produces a prediction:

$$
T_b(x), \quad b = 1, \dots, B
$$

The ensemble prediction is:

$$
\hat{f}_{RF}(x) = \frac{1}{B} \sum_{b=1}^{B} T_b(x)
$$

---

### II. Assumptions

1. Equal variance:

$$
\text{Var}(T_b(x)) = \sigma^2
$$

2. Constant correlation:

$$
\text{Corr}(T_i, T_j) = \rho
$$

3. Covariance:

$$
\text{Cov}(T_i, T_j) = \rho \sigma^2
$$

---

### III. Variance Expression

$$
\text{Var}(\hat{f}_{RF}) =
\text{Var}\left(\frac{1}{B} \sum_{b=1}^{B} T_b(x)\right)
$$

---

### IV. Expansion

$$
= \frac{1}{B^2}
\left(
\sum_{b=1}^{B} \sigma^2 +
\sum_{i \neq j} \rho \sigma^2
\right)
$$

---

### V. Counting Terms

$$
\sum \sigma^2 = B\sigma^2
$$

$$
\sum_{i \neq j} = B(B-1)
$$

---

### VI. Substitution

$$
\text{Var}(\hat{f}_{RF}) =
\frac{1}{B^2}
\left(
B\sigma^2 + B(B-1)\rho\sigma^2
\right)
$$

---

### VII. Simplification

$$
= \frac{\sigma^2}{B} (1 + (B-1)\rho)
$$

---

### VIII. Final Form

$$
\boxed{
\text{Var}(\text{RF}) =
\rho \sigma^2 + \frac{1 - \rho}{B}\sigma^2
}
$$

---

### IX. Special Case

If trees are independent:

$$
\rho = 0 \Rightarrow \text{Var} = \frac{\sigma^2}{B}
$$

---

### X. Interpretation

- $\rho \sigma^2$ → irreducible variance  
- $\frac{1-\rho}{B}\sigma^2$ → reducible via averaging  

---

## 12. Role of Feature Randomness

Feature subsampling reduces correlation:

$$
\rho \downarrow \Rightarrow \text{Var}(\text{RF}) \downarrow
$$

---

## 13. Bias–Variance Tradeoff

- Increasing trees:

$$
B \uparrow \Rightarrow \text{Variance} \downarrow
$$

- Feature randomness:

$$
\text{Variance} \downarrow
$$

- Bias remains low

---

## 14. Algorithm Summary

For $b = 1$ to $B$:

1. Sample dataset $D^{(b)}$  
2. Train tree with:
   - Random samples  
   - Random features  

Prediction:

$$
\hat{y} = \frac{1}{B} \sum_{b=1}^{B} T_b(x)
$$

---

## 15. Important

Random Forest Regressor combines:

- **Bagging**
- **Feature randomness**
- **Averaging**

to produce:

- Stable predictions  
- Low variance  
- Strong generalization  

---

## 16. Final Perspective

Random Forest approximates:

$$
\mathbb{E}_{D^{(b)}, \text{features}} [T(x)]
$$

i.e., an expectation over randomized trees, leading to robust regression.

In [1]:
class LeafNode:
    """
    Leaf node of the decision tree.

    Parameters
    ----------
    value : float
        The predicted value at the leaf node (mean or median of targets).
    """
    def __init__(self,value):
        self.value=value

In [2]:
class DecisionNode:
    """
    Internal decision node of the tree.

    Parameters
    ----------
    best_feature : int
        Index of the feature used for splitting.
    best_threshold : float
        Threshold value for the split.
    left_child : Node
        Left subtree (samples satisfying condition).
    right_child : Node
        Right subtree (samples not satisfying condition).
    """
    def __init__(self,best_feature, best_threshold , left_child, right_child):
        self.best_feature = best_feature
        self.best_threshold = best_threshold
        self.left_child = left_child
        self.right_child = right_child

In [3]:
class DecisionTreeRegressor:
    """
    Decision Tree Regressor supporting MSE, MAE, and Huber loss.

    Parameters
    ----------
    max_depth : int, default=10
        Maximum depth of the tree.

    feature_fraction : float, default=1.0
        Fraction of features randomly selected at each split.

    scoring : str, default='mse'
        Loss function used for splitting.
        Options: 'mse', 'mae', 'huber'

    min_sample_split : int, default=1
        Minimum number of samples required to split.

    delta : float, default=0.5
        Threshold parameter for Huber loss.

    Attributes
    ----------
    root : Node
        Root node of the decision tree.
    """
    def __init__(self , max_depth=1 , scoring ='mse' , feature_fraction = 1 , min_sample_split=1, delta=0.5):
        self.max_depth = max_depth
        self.scoring = scoring
        self.feature_fraction = feature_fraction
        self.min_sample_split = min_sample_split
        self.delta = delta
        self.root = None
        
        # Validate
        if type(self.max_depth) != int or self.max_depth <= 0:
            raise ValueError('Max depth must be a positive integer')
        if type(self.min_sample_split) != int or self.min_sample_split <= 0:
            raise ValueError('Min samples split must be a positive integer')
        if self.scoring not in ['mse', 'mae','huber']:
            raise ValueError("Scoring must be either 'mse', 'mae' or 'huber'")   
        if not (0 < self.feature_fraction <= 1):
            raise ValueError("feature_fraction must be in the range (0, 1].")
        if self.delta <= 0:
            raise ValueError("delta must be strictly greater than 0.")


    def _stopping_condition(self,data, depth):
        """
        Check whether to stop splitting.

        Stops if:
        - Maximum depth reached
        - Not enough samples
        - Target variance is zero
        """
        if depth >= self.max_depth:
            return True
        if len(data) < self.min_sample_split:
            return True
        if np.var(data[:,-1])==0:
            return True
        return False

    def _random_feature_sampling(self,data):
        """
        Randomly select a subset of features.

        Returns
        -------
        selected_features : array-like
            Indices of selected features.
        """
        num_features = data.shape[1]-1
        select_features = max(1,round(self.feature_fraction*num_features))
        selected_features = np.random.choice(num_features,select_features , replace=False)

        return selected_features

    def _leaf_prediction(self,y):
        """
        Compute prediction for a leaf node.

        Returns mean (MSE/HUBER) or median (MAE).
        """
        if self.scoring == 'mae':
            return np.median(y)
        else :
            return np.mean(y)

    def _all_thresholds(self,data , selected_features):
        """
        Generate candidate thresholds for each selected feature.
        """
        all_thresholds = []

        for feature in selected_features:
            unique_values = np.unique(data[:,feature])
            if len(unique_values) == 1:
                all_thresholds.append(np.array([]))
            else:
                # Midpoints between consecutive sorted values
                unique_averages = (unique_values[1:] + unique_values[:-1])/2
                all_thresholds.append(unique_averages)

        return all_thresholds

    def _split(self,data,feature , threshold):
        """
        Split dataset based on feature and threshold.
        """
        condition = data[:,feature] <= threshold

        return data[condition] , data[~condition]

    def _mse_loss(self,y):
        """Mean Squared Error (variance)."""
        return np.var(y)

    def _mae_loss(self,y):
        """Mean Absolute Error."""
        return np.mean(np.abs(y-np.median(y)))

    def _huber_loss(self,y):
        """Huber loss (robust to outliers)."""
        residual = y - np.mean(y)

        return np.mean(np.where(np.abs(residual) <= self.delta , 0.5 * (residual**2) , self.delta*(np.abs(residual)-0.5*self.delta)))

    def _score(self,left_y , right_y):
        """
        Compute weighted loss for a split.
        """
        n_left = len(left_y)
        n_right = len(right_y)

        if n_left ==0 or n_right==0:
            return np.inf

        total = n_left + n_right

        if self.scoring =='mse':
            left_score = self._mse_loss(left_y)
            right_score = self._mse_loss(right_y)

        elif self.scoring == 'mae':
            left_score = self._mae_loss(left_y)
            right_score = self._mae_loss(right_y) 

        elif self.scoring == 'huber':
            left_score = self._huber_loss(left_y)
            right_score = self._huber_loss(right_y)


        return (n_left*left_score + n_right*right_score)/total

    def _best_feature_threshold(self,data,all_thresholds,selected_features):
        """
        Find the best feature and threshold minimizing split loss.
        """
        best_feature = None
        best_threshold = None
        best_score = np.inf

        for i, feature in enumerate(selected_features):
            threshold_for_ith_feature = all_thresholds[i]
            if len(threshold_for_ith_feature)==0:
                continue

            else:

                for threshold in threshold_for_ith_feature:
                    left_data , right_data = self._split(data,feature,threshold)
                    score = self._score(left_data[:,-1],right_data[:,-1])
                    if score < best_score:
                        best_feature = feature 
                        best_score = score
                        best_threshold = threshold

        return best_feature , best_threshold

    def _find_best_split(self,data):
        """
        Select features and compute best split.
        """
        selected_feature = self._random_feature_sampling(data)
        all_thresholds = self._all_thresholds(data,selected_feature)
        best_feature , best_threshold = self._best_feature_threshold(data,all_thresholds,selected_feature)

        return best_feature , best_threshold

    def _build_tree(self,data, depth):
        """
        Recursively build the decision tree.
        """
        if self._stopping_condition(data,depth):
            return LeafNode(value = self._leaf_prediction(data[:,-1]))

        best_feature , best_threshold = self._find_best_split(data)

        if best_feature is None:
            return LeafNode(self._leaf_prediction(data[:,-1]))

        left_data , right_data = self._split(data ,best_feature ,  best_threshold)

        left_child = self._build_tree(left_data , depth+1)
        right_child = self._build_tree(right_data , depth+1)


        return DecisionNode(best_feature, best_threshold , left_child, right_child)

    def fit(self,X,y):
        """
        Train the decision tree.

        Parameters
        ----------
        X : array-like of shape (N, D)
            Feature matrix

        y : array-like of shape (N,)
            Target values
        """
        X = np.asarray(X)
        y = np.asarray(y)

        y = y.reshape(-1)

        if X.ndim==1:
            X = X.reshape(-1,1)

        data = np.column_stack((X,y))

        self.root  = self._build_tree(data,0)

    def _predict_single(self,x,node):
        """
        Predict output for a single sample.
        """
        if isinstance(node,LeafNode):
            return node.value
        if isinstance(node,DecisionNode):
            if x[node.best_feature] <= node.best_threshold :
                return self._predict_single(x, node.left_child)
            else:
                return self._predict_single(x, node.right_child)

    def predict(self,X):
        """
        Predict outputs for input samples.

        Parameters
        ----------
        X : array-like of shape (N, D)

        Returns
        -------
        predictions : ndarray
        """
        X = np.asarray(X)

        if X.ndim==1:
            X = X.reshape(-1,1)

        y_pred = np.array([self._predict_single(x,self.root)for x in X])

        return y_pred

In [4]:
class RandomForestRegressor:
    """
    Random Forest Regressor implemented from scratch using DecisionTreeRegressor as base learners.
    
    Parameters
    ----------
    n_trees : int
        Number of trees in the forest.
    max_depth : int
        Maximum depth of each decision tree.
    scoring : str
        Loss function to use for splitting ('mse', 'mae' or 'huber').
    feature_fraction : float
        Fraction of features to consider for each split (between 0 and 1).
    min_sample_split : int
        Minimum number of samples required to split a node.
    delta : float
        Delta parameter for Huber loss.
    sample_fraction : float
        Fraction of training samples used for each tree (bootstrap sampling).
    oob_score : bool
        Whether to compute out-of-bag (OOB) error.
    """
    def __init__(self,n_trees=100,max_depth=1,scoring ='mse',feature_fraction=1,
                 min_sample_split=1,delta=0.5,sample_fraction=1,oob_score=False):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.scoring = scoring
        self.feature_fraction = feature_fraction
        self.min_sample_split = min_sample_split
        self.delta = delta
        self.sample_fraction = sample_fraction
        self.oob_score = oob_score

        # Store OOB score if requested
        self.oob_mse_score = 0
        
        # List to store all trained decision trees
        self.all_trees =[]

    def _sampling(self,N):
        """
        Bootstrap sampling of dataset for a single tree.
        
        Parameters
        ----------
        N : int
            Total number of samples in the dataset.
        
        Returns
        -------
        sampled_indices : ndarray
            Indices of sampled data points for training.
        oob_indices : ndarray
            Indices of data points not included in this bootstrap sample (OOB).
        """
        num_samples = max(1,round(self.sample_fraction*N))
        all_indices = np.arange(N)
        # Sample with replacement
        sampled_indices = np.random.choice(all_indices ,num_samples , replace=True)
        oob_indices = np.setdiff1d(all_indices,sampled_indices)

        return sampled_indices,oob_indices

    def fit(self,X,y):
        """
        Fit the Random Forest Regressor to training data.
        
        Parameters
        ----------
        X : ndarray
            Feature matrix (N_samples x N_features).
        y : ndarray
            Target values (N_samples,).
        """

        X = np.asarray(X)
        y = np.asarray(y).reshape(-1, 1)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
    
        N = len(X)
        
        # Prepare OOB predictions if needed
        self.oob_preds = [[] for _ in range(N)]

        # Train each tree
        for i in range(self.n_trees):
            # Initialize a new DecisionTreeRegressor
            model = DecisionTreeRegressor(max_depth=self.max_depth,scoring=self.scoring,
                                          feature_fraction =self.feature_fraction, 
                                          min_sample_split=self.min_sample_split, delta=self.delta)

            # Bootstrap sample and get OOB indices
            sample_index , oob_index = self._sampling(N)

            X_sampled , y_sampled = X[sample_index,:], y[sample_index]

            # Bootstrap sample and get OOB indices
            model.fit(X_sampled , y_sampled)

            self.all_trees.append(model)

            # Store OOB predictions
            if self.oob_score:

                for oob_id in oob_index:
                    predict = model.predict(X[oob_id].reshape(1,-1))
                    
                    self.oob_preds[oob_id].append(predict)
                    
        # Compute OOB MSE if requested
        if self.oob_score:
            count =0
            for idx , predictions in enumerate(self.oob_preds):
                if len(predictions)>0:
                    y_mean = np.mean(predictions)
                    y_true = y[idx][0]
                    self.oob_mse_score += (y_mean-y_true)**2
                    count +=1
            print("OOB MSE Score:", self.oob_mse_score/count)

    def predict(self,X):
        """
        Predict target values for input features using the trained Random Forest.
        
        Parameters
        ----------
        X : ndarray
            Feature matrix (N_samples x N_features).
        
        Returns
        -------
        y_pred : ndarray
            Predicted target values (N_samples,).
        """
        X = np.asarray(X)
        if X.ndim == 1:
            X = X.reshape(-1,1)

        y_pred = np.array([model.predict(X) for model in self.all_trees])
        # Collect predictions from all trees
        return np.mean(y_pred,axis=0)       

# Random Forest Regression — Variance Reduction Experiment

## 1. Problem Setup

Generate a dataset with a nonlinear trend and some extreme outliers:

$$
y = \sin(x) + \epsilon, \quad \epsilon \sim \mathcal{N}(0, 0.5)
$$

Inject outliers to test robustness:

$$
y_{\text{outlier}} = y_{\text{outlier}} + 5
$$

The goal is to compare **variance of predictions** between:

- A single **Decision Tree Regressor (DT)**  
- A **Random Forest Regressor (RF)**  

across multiple **loss functions**.

---

## 2. Loss Functions Considered

Examine three types of regression losses:

1. **Mean Squared Error (MSE)**

$$
\text{MSE}(y, \hat{y}) = \frac{1}{N} \sum_{i=1}^N (y_i - \hat{y}_i)^2
$$

2. **Mean Absolute Error (MAE)**

$$
\text{MAE}(y, \hat{y}) = \frac{1}{N} \sum_{i=1}^N |y_i - \hat{y}_i|
$$

3. **Huber Loss** (with parameter $\delta$):

$$
L_\delta(y, \hat{y}) =
\begin{cases}
\frac{1}{2}(y-\hat{y})^2, & |y-\hat{y}| \le \delta \\
\delta (|y-\hat{y}| - \frac{1}{2}\delta), & |y-\hat{y}| > \delta
\end{cases}
$$

Huber interpolates between MSE and MAE, making it robust to outliers.

---

## 3. Setup

1. Generate **test set** $X_{\text{test}}$.  
2. For each loss function:
   - Train **Decision Tree** on bootstrapped samples ($M=30$ times)  
   - Train **Random Forest** (ensemble of $B=20$ trees)  
   - Collect predictions on $X_{\text{test}}$  
3. Compute variance of predictions across runs:

$$
\text{Var}_{DT} = \frac{1}{|X_{\text{test}}|} \sum_{i} \text{Var}(\hat{y}_i^{DT})
$$

$$
\text{Var}_{RF} = \frac{1}{|X_{\text{test}}|} \sum_{i} \text{Var}(\hat{y}_i^{RF})
$$

4. Compute **variance reduction**:

$$
\text{Variance Reduction} = \text{Var}_{DT} - \text{Var}_{RF}
$$

---


In [5]:
import numpy as np
import pandas as pd


# 1. Dataset

np.random.seed(42)
X = np.linspace(0, 10, 120).reshape(-1,1)
y = np.sin(X).flatten() + np.random.normal(0, 0.5, size=120)

# Add outliers
outlier_idx = np.random.choice(np.arange(120), size=8, replace=False)
y[outlier_idx] += 5

# Test set
X_test = np.linspace(0, 10, 60).reshape(-1,1)


# 2. Setup

M = 30  # number of runs

losses = {
    "MSE": {"scoring": "mse", "delta": 0.5},
    "MAE": {"scoring": "mae", "delta": 0.5},
    "Huber": {"scoring": "huber", "delta": 1.0}
}

In [6]:

results = []


# 3. Run Models

for loss_name, params in losses.items():

    dt_preds = []
    rf_preds = []

    for seed in range(M):
        np.random.seed(seed)

        
        # Decision Tree (bootstrap)
        
        indices = np.random.choice(len(X), size=len(X), replace=True)
        X_sample = X[indices]
        y_sample = y[indices]

        dt = DecisionTreeRegressor(
            max_depth=5,
            scoring=params["scoring"],
            delta=params["delta"]
        )
        dt.fit(X_sample, y_sample)
        dt_preds.append(dt.predict(X_test))

        
        # Random Forest
        
        rf = RandomForestRegressor(
            n_trees=20,
            max_depth=5,
            scoring=params["scoring"],
            delta=params["delta"]
        )
        rf.fit(X, y)
        rf_preds.append(rf.predict(X_test))

    # Convert to arrays
    dt_preds = np.array(dt_preds)
    rf_preds = np.array(rf_preds)

    # Compute variance across runs
    dt_variance = np.mean(np.var(dt_preds, axis=0))
    rf_variance = np.mean(np.var(rf_preds, axis=0))

    results.append({
        "Loss": loss_name,
        "DT Variance": round(dt_variance, 4),
        "RF Variance": round(rf_variance, 4),
        "Variance Reduction": round(dt_variance - rf_variance, 4)
    })


# 4. Results
df_results = pd.DataFrame(results)

## 4. Results

In [7]:
df_results

,Loss,DT Variance,RF Variance,Variance Reduction
0,MSE,0.8225,0.0527,0.7698
1,MAE,0.6104,0.0406,0.5699
2,Huber,0.7032,0.0462,0.6570


> **Observation:** Random Forest consistently reduces variance due to ensemble averaging effect.

---



## 5. Interpretation

1. **Variance Reduction:**  

$$
\text{Var}_{RF} < \text{Var}_{DT} \quad \forall \text{ losses}
$$

2. **Loss Sensitivity:**  
- **MSE** → sensitive to outliers → higher variance  
- **MAE** → robust → lower variance  
- **Huber** → intermediate behavior

3. **Theory Validation:**  
Random Forest variance follows:

$$
\text{Var}(\text{RF}) = \rho \sigma^2 + \frac{1-\rho}{B} \sigma^2
$$

- $\rho$ = correlation between trees  
- $B$ = number of trees  

Variance decreases as $B$ increases or $\rho$ decreases.

---

## 6. Conclusion

- Random Forest effectively **reduces variance** compared to a single tree.  
- Effect holds across **MSE, MAE and Huber losses**.  
- Ensemble methods are robust to outliers and stochasticity.  

---